In [ ]:
import numpy as np
import numpy as np
import random
np.random.seed(1)
simulation_result = []
scenarios = [
    {"n": 500, "p": 10, "z": 50},
    {"n": 500, "p": 50, "z": 5},
    {"n": 500, "p": 5, "z": 50},
    {"n": 500, "p": 50, "z": 200},
    {"n": 500, "p": 100, "z": 50},
    {"n": 500, "p": 100, "z": 100},
    {"n": 500, "p": 100, "z": 200},
    {"n": 500, "p": 200, "z": 50},
    {"n": 500, "p": 200, "z": 100},
    {"n": 500, "p": 200, "z": 200},
    {"n": 5000, "p": 100, "z": 100},
    {"n": 5000, "p": 100, "z": 500},
    {"n": 5000, "p": 100, "z": 1000},
    {"n": 5000, "p": 100, "z": 2000},
    {"n": 5000, "p": 500, "z": 100},
    {"n": 5000, "p": 500, "z": 500},
    {"n": 5000, "p": 500, "z": 1000},
    {"n": 5000, "p": 500, "z": 2000},
    {"n": 5000, "p": 1000, "z": 100},
    {"n": 5000, "p": 1000, "z": 500},
    {"n": 5000, "p": 1000, "z": 1000},
    {"n": 5000, "p": 1000, "z": 2000},

    {"n": 5000, "p": 2000, "z": 100},
    {"n": 5000, "p": 2000, "z": 500},
    {"n": 5000, "p": 2000, "z": 1000},
    {"n": 5000, "p": 2000, "z": 2000},

    # n = 10000
    {"n": 10000, "p": 500, "z": 500},
    {"n": 10000, "p": 500, "z": 1000},
    {"n": 10000, "p": 500, "z": 2000},

    {"n": 10000, "p": 1000, "z": 500},
    {"n": 10000, "p": 1000, "z": 1000},
    {"n": 10000, "p": 1000, "z": 2000},

    {"n": 10000, "p": 2000, "z": 500},
    {"n": 10000, "p": 2000, "z": 1000},
    {"n": 10000, "p": 2000, "z": 2000}
]

for s in scenarios:
        try:
            n, p, z = s["n"], s["p"], s["z"]
            n = n
            X = {}

            for i in range(1,p+1):
                if i % 5 == 1:
                    X[f"X{i}"] = np.random.normal(0,1,n)
                elif i % 5 == 2:
                    X[f"X{i}"] = np.random.uniform(0,1,n)
                elif i % 5 == 3:
                    X[f"X{i}"] = np.random.binomial(1, 0.5,n)
                elif i % 5 == 4:
                    X[f"X{i}"] = np.random.gamma(2, 1, n)
                else: 
                    X[f"X{i}"] = np.random.exponential(1, n)
            Z = {}
            for i in range(1,z+1):
                if i % 5 == 1:
                    Z[f"Z{i}"] = np.random.normal(0,1,n)
                elif i % 5 == 2:
                    Z[f"Z{i}"] = np.random.uniform(0,1,n)
                elif i % 5 == 3:
                    Z[f"Z{i}"] = np.random.binomial(1, 0.5,n)
                elif i % 5 == 4:
                    Z[f"Z{i}"] = np.random.gamma(2, 1, n)
                else: 
                    Z[f"Z{i}"] = np.random.exponential(1, n)
                    
                    
            import pandas as pd      
            X_df = pd.DataFrame(X)
            Z_df = pd.DataFrame(Z)

            XZ_df = pd.concat([X_df, Z_df], axis=1)       

            p_length = z + p

            beta0 = -1.0  # intercept (you choose this)
            np.random.seed(1)   
            beta = np.random.uniform(-10, 10, p_length)
            lin_pred = beta0 + XZ_df.values @ beta

            prob = 1 / (1 + np.exp(-lin_pred))

            Y = np.random.binomial(1, prob)
            y_bar = np.mean(Y)
            print(XZ_df)
            from sklearn.model_selection import train_test_split
            X_train, X_final_test, Y_train, Y_final_test = train_test_split(XZ_df,Y, test_size=0.2,random_state=10)

            import statsmodels.api as sm
            import pandas as pd

            results_list = []

            for col in X_train.columns:
                try:
                    X_temp = sm.add_constant(X_train[[col]])
                    
                    model = sm.Logit(Y_train, X_temp)
                    result = model.fit(disp=0)
                    
                    results_list.append({
                        "variable": col,
                        "coef": result.params[col],
                        "p_value": result.pvalues[col]
                    })
                    
                except Exception as e:
                    # skip problematic variables
                    print(f"Skipping {col}: {e}")

            univariate_results = pd.DataFrame(results_list).sort_values(by="p_value")

            print(univariate_results)


            sig_vars = univariate_results.loc[univariate_results["p_value"] < 0.05, "variable"].tolist()


            print(sig_vars)


            X_train_sig = X_train[sig_vars]




            # Further splitting currrent data to train-test. 


            from sklearn.ensemble import RandomForestClassifier
            from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
            from scipy.stats import randint

            rf = RandomForestClassifier(random_state=42)

            param_dist = {
                'n_estimators': randint(100, 400),
                'max_features': ['sqrt', 'log2', 10, 20],
                'min_samples_leaf': randint(1, 4),
                'min_samples_split': randint(2, 10)
            }

            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

            search = RandomizedSearchCV(
                rf,
                param_distributions=param_dist,
                n_iter=30,
                scoring='roc_auc',
                cv=cv,
                n_jobs=-1,
                random_state=42
            )

            # 🔥 ONLY training data here
            search.fit(X_train[sig_vars], Y_train)

            print("Best params:", search.best_params_)
            print("Best CV AUC:", search.best_score_)
            best_rf = search.best_estimator_
            from sklearn.inspection import permutation_importance

            mda = permutation_importance(
                best_rf,
                X_final_test[sig_vars],
                Y_final_test,
                n_repeats=10,
                scoring='roc_auc',   
                random_state=42
                )

            mda_df = pd.DataFrame({
                "feature": sig_vars,
                "MeanDecreaseAccuracy": mda.importances_mean
            }).sort_values(by="MeanDecreaseAccuracy", ascending=False)
            print(mda_df)



            y_final_prob = best_rf.predict_proba(X_final_test[sig_vars])[:, 1]

            from sklearn.metrics import roc_auc_score

            traditional_final_auc = roc_auc_score(Y_final_test, y_final_prob)

            print("Final AUC:", traditional_final_auc)




            # Interpretablity



            np.random.seed(1)  # for reproducibility

            

            X_inter = pd.DataFrame({
                "variable": [f"X{i}" for i in range(1, p+1)],
                "value": np.random.randint(51,101,p)
            })

            Z_inter = pd.DataFrame({
                "variable": [f"Z{i}" for i in range(1, p+1)],
                "value":np.random.randint(1, 51,p)
            })


            inter = pd.concat([X_inter, Z_inter], axis=0)


            inter_filtered = inter[inter["variable"].isin(sig_vars)]

            merged_df = inter_filtered.merge(
                univariate_results,
                on="variable",
                how="left"
            )

            mda_df["weight"] = mda_df["MeanDecreaseAccuracy"]/mda_df["MeanDecreaseAccuracy"].sum()
            traditional_sum_inter = merged_df["value"].sum()
            traditional_mean_inter = merged_df["value"].mean()
            weight_inter = mda_df.merge(merged_df, left_on="feature", right_on="variable",how = "left")
            weight_inter["weighted_imp"] = weight_inter["value"] * weight_inter["weight"]
            traditional_w_inter_sum = weight_inter["weighted_imp"].sum()
            traditional_w_inter_mean = weight_inter["weighted_imp"].mean()
            # This is the curve function
            def curve_fun(d, x, t, y_min=0.02, max_y=1):
                x = np.array(x)  # ensure numpy array
                x = x[::-1]
                p = np.max(x)
                x_norm = x / p
                
                # Log and exponential components
                log_t = 1 - np.log(1 + d * x_norm) / np.log(1 + d)
                exp_t = 1 - (np.exp(d * x_norm) - 1) / (np.exp(d) - 1)
                
                # Weighted combination
                ans = t * log_t + (1 - t) * exp_t
                
                # Rescale to [y_min, max_y]
                ans = y_min + (ans - np.min(ans)) / (np.max(ans) - np.min(ans)) * (max_y - y_min)
                
                return ans


            results = []

            for d in range(1, 11):
                for t in [0, 0.5, 1]:
                    temp_df = merged_df.copy()
                    temp_df["threshold"] = curve_fun(d=d, t=t, x=temp_df["value"]) * 0.05
                    after_int_sig_vars = temp_df.loc[temp_df["p_value"] < temp_df["threshold"], "variable"].tolist()
                    after_int_sig_vars = [ v for v in after_int_sig_vars if v in X_train.columns and v in X_final_test.columns]
                    if len(after_int_sig_vars) == 0:
                        continue
                    rf2 = RandomForestClassifier(random_state=42)
                    param_dist = {
                        'n_estimators': randint(100, 400),
                        'max_features': ['sqrt', 'log2', 10, 20],
                        'min_samples_leaf': randint(1, 4),
                        'min_samples_split': randint(2, 10)
                    }
                    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                    search = RandomizedSearchCV(
                        rf2,
                        param_distributions=param_dist,
                        n_iter=30,
                        scoring='roc_auc',
                        cv=cv,
                        n_jobs=-1,
                        random_state=42
                    )
                    search.fit(X_train[after_int_sig_vars], Y_train)
                    best_rf = search.best_estimator_
                    y_final_prob = best_rf.predict_proba(
                        X_final_test[after_int_sig_vars]
                    )[:, 1]
                    final_auc = roc_auc_score(Y_final_test, y_final_prob)
                    results.append({
                        "d": d,
                        "t": t,
                        "n_vars": len(after_int_sig_vars),
                        "cv_auc": search.best_score_,
                        "test_auc": final_auc,
                        "n_estimators": search.best_params_["n_estimators"],
                        "max_features": search.best_params_["max_features"],
                        "min_samples_leaf": search.best_params_["min_samples_leaf"],
                        "min_samples_split": search.best_params_["min_samples_split"],
                        "model": best_rf,
                        "vars": after_int_sig_vars
                    })
                    
            df_results = pd.DataFrame(results)
            best_row = df_results.loc[df_results["test_auc"].idxmax()]        
            best_d = best_row["d"]
            best_t = best_row["t"]
            merged_df["new_threshold"] = curve_fun(d=best_d, t=best_t, x=temp_df["value"]) * 0.05
            best_vars = (merged_df[merged_df["p_value"] < merged_df["new_threshold"]])["variable"].tolist()

            best = max(results, key=lambda x: x["test_auc"])

            best_model = best["model"]
            best_vars = best["vars"]
            y_final_prob = best_model.predict_proba(X_final_test[best_vars])[:, 1]

            new_mda = permutation_importance(
                best_model,
                X_final_test[best_vars],
                Y_final_test,
                n_repeats=10,
                scoring='roc_auc',   
                random_state=42
                )

            new_mda_df = pd.DataFrame({
                "feature": best_vars,
                "MeanDecreaseAccuracy": new_mda.importances_mean
            }).sort_values(by="MeanDecreaseAccuracy", ascending=False)

            new_final_auc = roc_auc_score(Y_final_test, y_final_prob)
            new_merged = merged_df[merged_df["variable"].isin(best_vars)]
            new_mean_inter = new_merged["value"].mean()
            new_sum_inter =  new_merged["value"].sum()

            new_mda_df["weight"] = new_mda_df["MeanDecreaseAccuracy"] / new_mda_df["MeanDecreaseAccuracy"].sum()
            new_weight_inter = new_mda_df.merge(new_merged, left_on="feature", right_on="variable", how = "left")
            new_weight_inter["weighted_imp"] = new_weight_inter["value"] * new_weight_inter["weight"]


            new_w_inter_sum = new_weight_inter["weighted_imp"].sum()
            new_w_inter_mean = new_weight_inter["weighted_imp"].mean()




            print(traditional_final_auc)
            print(new_final_auc)
            print(traditional_mean_inter)
            print(new_mean_inter)
            print(traditional_sum_inter)
            print(new_sum_inter)
            print(traditional_w_inter_sum)
            print(new_w_inter_sum)
            print(traditional_w_inter_mean)
            print(new_w_inter_mean)
            results_table = pd.DataFrame([{
            "traditional_auc": traditional_final_auc,
            "new_auc": new_final_auc,

            "traditional_mean_inter": traditional_mean_inter,
            "new_mean_inter": new_mean_inter,

            "traditional_sum_inter": traditional_sum_inter,
            "new_sum_inter": new_sum_inter,

            "traditional_w_inter_sum": traditional_w_inter_sum,
            "new_w_inter_sum": new_w_inter_sum,

            "traditional_w_inter_mean": traditional_w_inter_mean,
            "new_w_inter_mean": new_w_inter_mean,
            
            "d" : best_d,
            "t" : best_t,
            "n" : n,
            "z" : z,
            "p" : p ,
            "num_var_ori" : len(sig_vars),
            "num_var_new" : len(best_vars)
            
            }])


            print(results_table)
            print(d)
            print(t)
            print("Done for 1 cycle")
        except Exception as e:
            print(f"Skipping n={n}, p={p}, z={z} due to error: {e}")
            continue
df = pd.DataFrame(results_table)
df.to_csv("simulation_results.csv", index=False)


In [ ]:
print(df)